# Multi-modal, Multi-level QCELS

This example estimates the two hydrogen molecule eigenvalues with nonzero Hartree--Fock overlap. It uses the repository's frozen molecular-integral snapshot. The same problem is evaluated in classical, exact StateVector, and exact Hadamard-test modes, followed by the opt-in `error_rate` parameter mode.

In [ ]:
from pathlib import Path

import numpy as np

import qarp
from qarp.algorithms import MMQCELS
from qarp.blocks import ComputationalBasisStateBlock, TrotterBlock
from qarp.engines import QarpEngine
from qarp.operators import JordanWigner
from qarp.operators.integrals import restricted_integrals_to_fermion_operator
from qarp.operators.onv import onv_from_spatial_occupations

asset_relative_path = Path("tests/assets/molecules/h2_0.735_sto3g.npz")
repo_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / asset_relative_path).is_file()
)
with np.load(repo_root / asset_relative_path) as data:
    fermion_operator = restricted_integrals_to_fermion_operator(
        float(data["constant"]), data["one_electron"], data["two_electron"]
    )

mapping = JordanWigner()
hamiltonian = mapping.encode_operator(fermion_operator)
onv = onv_from_spatial_occupations([2, 0])
state = ComputationalBasisStateBlock(mapping.encode_state(onv)).build()
reference = np.array([-1.1373060357533997, 0.4950577416181092])

common = dict(
    state=state,
    T0=1.25,
    N0=120,
    Nj=80,
    n_dominant_eigenvalues=2,
    initial_eigenvalues=[-1.0, 0.5],
    n_initial_guesses=10,
    seed=1,
    verbose=False,
)

## Standard parameter mode

The default `standard` mode requires `T0`, `N0`, `Nj`, and exactly one of `n_levels` or `q`. First, exact classical evolution provides the reference MMQCELS fit.

In [ ]:
classical = MMQCELS(
    operator=hamiltonian,
    execution_mode="classical",
    n_levels=5,
    **common,
)
classical_eigenvalues = classical.run()
classical_error = np.abs(classical_eigenvalues - reference)
assert np.max(classical_error) < 1e-5
{
    "eigenvalues": classical_eigenvalues,
    "amplitudes": classical.amplitudes,
    "absolute error": classical_error,
}

## Circuit execution modes

Circuit modes require a symbolic-time `TrotterBlock`. The StateVector calculation is exact for that Trotterized circuit, so its error against the molecular reference includes the finite-order Trotter approximation.

In [ ]:
trotter = TrotterBlock(state.n_qubits, hamiltonian, steps=24, order=2).build()
statevector = MMQCELS(
    operator=trotter,
    execution_mode="statevector",
    n_levels=4,
    engine=QarpEngine(seed=1),
    **common,
)
statevector_eigenvalues = statevector.run()
statevector_error = np.abs(statevector_eigenvalues - reference)
assert np.max(statevector_error) < 6e-3
{
    "eigenvalues": statevector_eigenvalues,
    "amplitudes": statevector.amplitudes,
    "absolute error (fit + Trotter)": statevector_error,
}

In [ ]:
hadamard = MMQCELS(
    operator=trotter,
    execution_mode="hadamard",
    n_shots=qarp.EXACT,
    n_levels=4,
    engine=QarpEngine(seed=1),
    **common,
)
hadamard_eigenvalues = hadamard.run()
assert np.allclose(hadamard_eigenvalues, statevector_eigenvalues, atol=1e-8)
assert np.allclose(hadamard.amplitudes, statevector.amplitudes, atol=1e-8)
{
    "eigenvalues": hadamard_eigenvalues,
    "amplitudes": hadamard.amplitudes,
    "absolute error (fit + Trotter)": np.abs(hadamard_eigenvalues - reference),
}

## Error-rate-derived parameter mode

The opt-in `error_rate` mode retains OpenQARP's epsilon-only level and sample-count formulas. `T0` is still required because an error rate cannot determine a dimensionful initial time scale. Explicit `N0`, `Nj`, or `n_levels` values may override the derived values.

In [ ]:
error_rate_mode = MMQCELS(
    operator=hamiltonian,
    state=state,
    execution_mode="classical",
    parameter_mode="error_rate",
    T0=0.1,
    error_rate=1e-2,
    n_dominant_eigenvalues=2,
    initial_eigenvalues=[-1.0, 0.5],
    seed=1,
    verbose=False,
)
assert (error_rate_mode.n_levels, error_rate_mode.N0, error_rate_mode.Nj) == (8, 5, 10)
error_rate_eigenvalues = error_rate_mode.run()
error_rate_error = np.abs(error_rate_eigenvalues - reference)
assert np.max(error_rate_error) < error_rate_mode.error_rate
{
    "derived schedule (n_levels, N0, Nj)": (
        error_rate_mode.n_levels,
        error_rate_mode.N0,
        error_rate_mode.Nj,
    ),
    "eigenvalues": error_rate_eigenvalues,
    "amplitudes": error_rate_mode.amplitudes,
    "absolute error": error_rate_error,
}